In [2]:
# RetailPulse — KartZone India Pvt. Ltd.

##  ORDERS TABLE GENERATOR

#
# PURPOSE:
# Generate 10,000 realistic orders using the FINAL cleaned Customer and Product
# master tables.
#
# IMPORTANT FIX:
# Product_IDs are now selected from KartZone_Products_Final.csv instead of
# artificially generating product IDs from numeric ranges.
#
# This prevents thousands of false invalid Product_ID foreign keys.
#
# DATA QUALITY ISSUES INTENTIONALLY ADDED:
#   • Mixed date formats
#   • Currency symbols in Final_Amount
#   • Inconsistent Order_Status values
#   • Inconsistent Category names
#   • Inconsistent Payment_Mode
#   • Negative Final_Amount
#   • Quantity as string/float/int
#   • Missing Expected_Delivery_Date
#   • Missing Return_Reason, Coupon_Code, Profit
#   • Duplicate order rows
#   • Blank rows
#   • Invalid discount values
#   • Approximately 2% orphan Customer_IDs
#   • Approximately 2% orphan Product_IDs
#
# ═══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import warnings

warnings.filterwarnings('ignore')

np.random.seed(44)
random.seed(44)

print("=" * 70)
print("  KARTZONE INDIA — ORDERS TABLE GENERATOR")
print("=" * 70)


# ═══════════════════════════════════════════════════════════════════════════════
# 1. LOAD VALID MASTER DATA
# ═══════════════════════════════════════════════════════════════════════════════

print("\nLoading cleaned master tables...")

customers_master = pd.read_csv('KartZone_Customers_Final.csv')
products_master = pd.read_csv('KartZone_Products_Final.csv')

# Clean IDs
customers_master['Customer_ID'] = (
    customers_master['Customer_ID']
    .astype('string')
    .str.strip()
)

products_master['Product_ID'] = (
    products_master['Product_ID']
    .astype('string')
    .str.strip()
)

# Remove missing IDs just in case
customers_master = customers_master[
    customers_master['Customer_ID'].notna()
].copy()

products_master = products_master[
    products_master['Product_ID'].notna()
].copy()

# Unique valid IDs
CUSTOMER_IDS = customers_master['Customer_ID'].drop_duplicates().tolist()
PRODUCT_IDS = products_master['Product_ID'].drop_duplicates().tolist()

print(f"Valid Customer IDs loaded : {len(CUSTOMER_IDS):,}")
print(f"Valid Product IDs loaded  : {len(PRODUCT_IDS):,}")


# ═══════════════════════════════════════════════════════════════════════════════
# 2. PRODUCT MASTER LOOKUP
# ═══════════════════════════════════════════════════════════════════════════════

# Create Product_ID -> Category lookup.
#
# Your Products_Final file should contain Category.
# If the exact category column is available, use it.
# Otherwise the generator will fall back to random category selection.

PRODUCT_CATEGORY_MAP = {}

if 'Category' in products_master.columns:

    product_category_df = products_master[
        ['Product_ID', 'Category']
    ].copy()

    product_category_df['Category'] = (
        product_category_df['Category']
        .astype('string')
        .str.strip()
    )

    PRODUCT_CATEGORY_MAP = dict(
        zip(
            product_category_df['Product_ID'],
            product_category_df['Category']
        )
    )

    print("Product category lookup created.")

else:
    print(
        "WARNING: Category column not found in Products_Final.csv."
    )
    print(
        "Orders will use randomly selected categories."
    )


# ═══════════════════════════════════════════════════════════════════════════════
# 3. CATEGORY DEFINITIONS
# ═══════════════════════════════════════════════════════════════════════════════

CANONICAL_CATEGORIES = [
    'Electronics',
    'Fashion',
    'Home & Kitchen',
    'Beauty'
]

CAT_W = [35, 30, 20, 15]

CAT_VARIANTS = {
    'Electronics': [
        'Electronics',
        'electronics',
        'ELECTRONICS',
        'Electronis',
        'Electronic'
    ],

    'Fashion': [
        'Fashion',
        'fashion',
        'FASHION',
        'Fasion',
        'Clothing'
    ],

    'Home & Kitchen': [
        'Home & Kitchen',
        'home & kitchen',
        'HOME & KITCHEN',
        'Home',
        'H&K'
    ],

    'Beauty': [
        'Beauty',
        'beauty',
        'BEAUTY',
        'Beuty',
        'Personal Care'
    ]
}


# ═══════════════════════════════════════════════════════════════════════════════
# 4. CITY DEFINITIONS
# ═══════════════════════════════════════════════════════════════════════════════

CITY_VARIANTS = {

    'Mumbai': [
        'Mumbai',
        'mumbai',
        'MUMBAI',
        'Bombay',
        'mumbai '
    ],

    'Delhi': [
        'Delhi',
        'delhi',
        'New Delhi',
        'NEW DELHI',
        'Delhi '
    ],

    'Bangalore': [
        'Bangalore',
        'bangalore',
        'Bengaluru',
        'BANGALORE',
        'Banglore'
    ],

    'Chennai': [
        'Chennai',
        'chennai',
        'CHENNAI',
        'Madras',
        'chennai '
    ],

    'Hyderabad': [
        'Hyderabad',
        'hyderabad',
        'HYDERABAD',
        'Hyd',
        'Hydrabad'
    ],

    'Pune': [
        'Pune',
        'pune',
        'PUNE',
        'Poona',
        'pune '
    ]
}

CITY_W = {
    'Mumbai': 25,
    'Delhi': 22,
    'Bangalore': 20,
    'Chennai': 13,
    'Hyderabad': 12,
    'Pune': 8
}


# ═══════════════════════════════════════════════════════════════════════════════
# 5. ORDER STATUS
# ═══════════════════════════════════════════════════════════════════════════════

STATUS_V = {

    'Delivered': [
        'Delivered',
        'delivered',
        'DELIVERED',
        'Deliverd',
        'Complete',
        'Completed'
    ],

    'Returned': [
        'Returned',
        'returned',
        'RETURNED',
        'Return',
        'Refunded',
        'refunded'
    ],

    'Cancelled': [
        'Cancelled',
        'cancelled',
        'CANCELLED',
        'Canceled',
        'Cancel'
    ],

    'Pending': [
        'Pending',
        'pending',
        'PENDING',
        'In Transit',
        'Processing',
        'Shipped'
    ]
}


# ═══════════════════════════════════════════════════════════════════════════════
# 6. PAYMENT MODE
# ═══════════════════════════════════════════════════════════════════════════════

PAYMENT_V = {

    'UPI': [
        'UPI',
        'upi',
        'UPI Payment',
        'Upi',
        'UPI Transfer'
    ],

    'Credit Card': [
        'Credit Card',
        'credit card',
        'CREDIT CARD',
        'CC',
        'CreditCard'
    ],

    'Debit Card': [
        'Debit Card',
        'debit card',
        'DEBIT CARD',
        'DC',
        'DebitCard'
    ],

    'COD': [
        'COD',
        'cod',
        'Cash on Delivery',
        'Cash',
        'cash on delivery'
    ],

    'Net Banking': [
        'Net Banking',
        'net banking',
        'NET BANKING',
        'NetBanking',
        'NB'
    ],

    'EMI': [
        'EMI',
        'emi',
        'Easy EMI',
        'No Cost EMI',
        'emi scheme'
    ],

    'Wallet': [
        'Wallet',
        'wallet',
        'PhonePe',
        'Paytm',
        'Amazon Pay'
    ]
}

# UPI dominant
PAYMENT_W = [35, 15, 12, 18, 8, 7, 5]


# ═══════════════════════════════════════════════════════════════════════════════
# 7. RETURN REASONS
# ═══════════════════════════════════════════════════════════════════════════════

RETURN_REASONS = [

    'Product Damaged',
    'Wrong Item Delivered',
    'Quality Not as Expected',
    'Changed Mind',
    'Size/Fit Issue',
    'Delayed Delivery',
    'Product Defective',
    'Not as Described',
    'Better Price Available',
    'Duplicate Order',

    None,
    None,
    None
]


# ═══════════════════════════════════════════════════════════════════════════════
# 8. COUPON CODES
# ═══════════════════════════════════════════════════════════════════════════════

COUPON_CODES = [

    'SAVE10',
    'FEST20',
    'WELCOME15',
    'APP25',
    'NEWUSER10',
    'SALE30',
    'KARTZONE15',
    'FLASH20',
    'DIWALI25',
    'NEWYEAR10',

    None,
    None,
    None,
    None
]


# ═══════════════════════════════════════════════════════════════════════════════
# 9. SEASONAL WEIGHTS
# ═══════════════════════════════════════════════════════════════════════════════

MONTH_W = {
    1: 5,
    2: 4,
    3: 5,
    4: 5,
    5: 5,
    6: 4,
    7: 5,
    8: 5,
    9: 6,
    10: 12,
    11: 15,
    12: 10
}


# ═══════════════════════════════════════════════════════════════════════════════
# 10. CATEGORY PRICING
# ═══════════════════════════════════════════════════════════════════════════════

COST_RANGES = {

    'Electronics': (
        1500,
        70000
    ),

    'Fashion': (
        150,
        4500
    ),

    'Home & Kitchen': (
        200,
        7000
    ),

    'Beauty': (
        80,
        1800
    )
}


MARKUP = {

    'Electronics': (
        1.20,
        1.60
    ),

    'Fashion': (
        1.50,
        3.00
    ),

    'Home & Kitchen': (
        1.30,
        2.00
    ),

    'Beauty': (
        1.40,
        2.50
    )
}


# ═══════════════════════════════════════════════════════════════════════════════
# 11. HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

def messy_dt(dt):

    return dt.strftime(
        random.choice([
            '%Y-%m-%d',
            '%d/%m/%Y',
            '%d-%m-%Y',
            '%m/%d/%Y',
            '%d.%m.%Y'
        ])
    )


def messy_amount(amt):

    return random.choices(

        [
            amt,
            f'₹{amt}',
            f'Rs.{amt}',
            f'INR {amt}',
            str(amt)
        ],

        weights=[
            55,
            15,
            12,
            8,
            10
        ]
    )[0]


def messy_qty(q):

    return random.choices(

        [
            q,
            float(q),
            str(q),
            None
        ],

        weights=[
            65,
            12,
            15,
            8
        ]
    )[0]


# ═══════════════════════════════════════════════════════════════════════════════
# 12. GENERATE ORDERS
# ═══════════════════════════════════════════════════════════════════════════════

print("\nGenerating 10,000 order records...")

orders = []

order_num = 100001


# Build seasonal month pool
month_pool = []

for m, w in MONTH_W.items():

    month_pool.extend(
        [m] * w
    )


for _ in range(10000):

    # ─────────────────────────────────────────────────────────────────────────
    # Order ID
    # ─────────────────────────────────────────────────────────────────────────

    oid = f'ORD-KZ-{order_num}'

    order_num += 1


    # ─────────────────────────────────────────────────────────────────────────
    # Order Date
    # ─────────────────────────────────────────────────────────────────────────

    month = random.choice(month_pool)

    year = random.choices(
        [2023, 2024],
        weights=[50, 50]
    )[0]

    day = random.randint(1, 28)

    order_date = datetime(
        year,
        month,
        day
    )


    # ─────────────────────────────────────────────────────────────────────────
    # Expected Delivery
    # ─────────────────────────────────────────────────────────────────────────

    exp_days = random.randint(3, 7)

    exp_del = (
        order_date +
        timedelta(days=exp_days)
    )


    # ─────────────────────────────────────────────────────────────────────────
    # Customer
    # ─────────────────────────────────────────────────────────────────────────

    cid = random.choice(
        CUSTOMER_IDS
    )

    # Approximately 2% intentionally invalid customers
    if random.random() < 0.02:

        cid = f'C{random.randint(9000, 9999)}'


    # ─────────────────────────────────────────────────────────────────────────
    # Product
    # ─────────────────────────────────────────────────────────────────────────
    #
    # IMPORTANT:
    # Use Product_IDs from the actual cleaned Products master.
    #
    # This is the main correction from your old generator.
    # ─────────────────────────────────────────────────────────────────────────

    pid = random.choice(
        PRODUCT_IDS
    )

    # Approximately 2% intentionally invalid products
    if random.random() < 0.02:

        pid = f'INVALID-{random.randint(1, 9999)}'


    # ─────────────────────────────────────────────────────────────────────────
    # Determine Category
    # ─────────────────────────────────────────────────────────────────────────

    if pid in PRODUCT_CATEGORY_MAP:

        original_category = PRODUCT_CATEGORY_MAP[pid]

        category_text = str(
            original_category
        ).strip().lower()

        if 'electron' in category_text:

            cat_key = 'Electronics'

        elif (
            'fashion' in category_text
            or 'cloth' in category_text
        ):

            cat_key = 'Fashion'

        elif (
            'home' in category_text
            or 'kitchen' in category_text
        ):

            cat_key = 'Home & Kitchen'

        elif 'beauty' in category_text:

            cat_key = 'Beauty'

        else:

            cat_key = random.choices(
                CANONICAL_CATEGORIES,
                weights=CAT_W
            )[0]

    else:

        # Invalid product gets a category so the row can still be generated
        cat_key = random.choices(
            CANONICAL_CATEGORIES,
            weights=CAT_W
        )[0]


    # Add intentional category variation
    cat_val = random.choice(
        CAT_VARIANTS[cat_key]
    )


    # ─────────────────────────────────────────────────────────────────────────
    # City
    # ─────────────────────────────────────────────────────────────────────────

    city_key = random.choices(

        list(CITY_W.keys()),

        weights=list(
            CITY_W.values()
        )

    )[0]

    city_val = random.choice(
        CITY_VARIANTS[city_key]
    )


    # ─────────────────────────────────────────────────────────────────────────
    # Pricing
    # ─────────────────────────────────────────────────────────────────────────

    cmin, cmax = COST_RANGES[cat_key]

    unit_cost = round(
        random.uniform(
            cmin,
            cmax
        ),
        2
    )

    mmin, mmax = MARKUP[cat_key]

    mrp = round(
        unit_cost *
        random.uniform(
            mmin,
            mmax
        ),
        2
    )


    # ─────────────────────────────────────────────────────────────────────────
    # Discount
    # ─────────────────────────────────────────────────────────────────────────

    if cat_key == 'Electronics':

        disc = random.choices(

            [0, 5, 10, 15, 20, 25, 30, 35, 40],

            weights=[
                10, 10, 15, 20, 20,
                12, 8, 3, 2
            ]

        )[0]

    elif cat_key == 'Fashion':

        disc = random.choices(

            [0, 5, 10, 15, 20, 25, 30, 40, 50],

            weights=[
                10, 10, 15, 20, 15,
                12, 8, 5, 5
            ]

        )[0]

    elif cat_key == 'Beauty':

        disc = random.choices(

            [0, 5, 10, 15, 20, 25, 30],

            weights=[
                20, 20, 20, 15,
                12, 8, 5
            ]

        )[0]

    else:

        disc = random.choices(

            [0, 5, 10, 15, 20],

            weights=[
                30, 25, 25, 12, 8
            ]

        )[0]


    # ─────────────────────────────────────────────────────────────────────────
    # Quantity and Amounts
    # ─────────────────────────────────────────────────────────────────────────

    qty = random.choices(

        [1, 2, 3, 4, 5],

        weights=[
            50, 25, 12, 8, 5
        ]

    )[0]


    selling_price = round(
        mrp *
        (1 - disc / 100),
        2
    )


    final_amount = round(
        selling_price * qty,
        2
    )


    total_cost = round(
        unit_cost * qty,
        2
    )


    profit = round(
        final_amount - total_cost,
        2
    )


    profit_margin = round(

        (
            profit /
            final_amount *
            100
        )
        if final_amount > 0
        else 0,

        2
    )


    # ─────────────────────────────────────────────────────────────────────────
    # Payment
    # ─────────────────────────────────────────────────────────────────────────

    pay_key = random.choices(

        list(PAYMENT_V.keys()),

        weights=PAYMENT_W

    )[0]

    pay_val = random.choice(
        PAYMENT_V[pay_key]
    )


    # ─────────────────────────────────────────────────────────────────────────
    # Order Status
    # ─────────────────────────────────────────────────────────────────────────

    if pay_key == 'COD':

        status_key = random.choices(

            [
                'Delivered',
                'Returned',
                'Cancelled',
                'Pending'
            ],

            weights=[
                55,
                15,
                22,
                8
            ]

        )[0]

    elif (
        cat_key == 'Fashion'
        and exp_days > 5
    ):

        status_key = random.choices(

            [
                'Delivered',
                'Returned',
                'Cancelled',
                'Pending'
            ],

            weights=[
                50,
                30,
                12,
                8
            ]

        )[0]

    elif (
        cat_key == 'Electronics'
        and disc > 25
    ):

        status_key = random.choices(

            [
                'Delivered',
                'Returned',
                'Cancelled',
                'Pending'
            ],

            weights=[
                60,
                20,
                12,
                8
            ]

        )[0]

    else:

        status_key = random.choices(

            [
                'Delivered',
                'Returned',
                'Cancelled',
                'Pending'
            ],

            weights=[
                70,
                12,
                10,
                8
            ]

        )[0]


    status_val = random.choice(
        STATUS_V[status_key]
    )


    # ─────────────────────────────────────────────────────────────────────────
    # Return Reason
    # ─────────────────────────────────────────────────────────────────────────

    ret_reason = None

    if status_key == 'Returned':

        ret_reason = random.choice(
            RETURN_REASONS
        )


    # ─────────────────────────────────────────────────────────────────────────
    # Coupon
    # ─────────────────────────────────────────────────────────────────────────

    coupon = random.choice(
        COUPON_CODES
    )


    # ─────────────────────────────────────────────────────────────────────────
    # Intentional Final Amount Mess
    # ─────────────────────────────────────────────────────────────────────────

    final_fmt = random.choices(

        [
            final_amount,
            messy_amount(final_amount),
            -final_amount,
            0
        ],

        weights=[
            60,
            25,
            8,
            7
        ]

    )[0]


    # ─────────────────────────────────────────────────────────────────────────
    # Intentional Profit Mess
    # ─────────────────────────────────────────────────────────────────────────

    profit_fmt = random.choices(

        [
            profit,
            None,
            profit * -1,
            0,
            str(round(profit, 2))
        ],

        weights=[
            65,
            12,
            8,
            8,
            7
        ]

    )[0]


    # ─────────────────────────────────────────────────────────────────────────
    # Intentional Discount Mess
    # ─────────────────────────────────────────────────────────────────────────

    disc_fmt = random.choices(

        [
            disc,
            float(disc),
            f'{disc}%',
            -disc,
            disc + random.randint(1, 50),
            None
        ],

        weights=[
            55,
            10,
            15,
            6,
            6,
            8
        ]

    )[0]


    # ─────────────────────────────────────────────────────────────────────────
    # Append Order
    # ─────────────────────────────────────────────────────────────────────────

    orders.append({

        'Order_ID':
            oid,

        'Customer_ID':
            cid,

        'Product_ID':
            pid,

        'Order_Date':
            messy_dt(order_date),

        'Expected_Delivery_Date':
            random.choices(

                [
                    messy_dt(exp_del),
                    None,
                    'NA'
                ],

                weights=[
                    82,
                    12,
                    6
                ]

            )[0],

        'Category':
            cat_val,

        'City':
            city_val,

        'Payment_Mode':
            pay_val,

        'Quantity':
            messy_qty(qty),

        'MRP':
            round(mrp, 2),

        'Discount_Pct':
            disc_fmt,

        'Selling_Price':
            round(selling_price, 2),

        'Final_Amount':
            final_fmt,

        'Unit_Cost':
            round(unit_cost, 2),

        'Total_Cost':
            round(total_cost, 2),

        'Profit':
            profit_fmt,

        'Profit_Margin_Pct':
            profit_margin,

        'Order_Status':
            status_val,

        'Return_Reason':
            ret_reason,

        'Coupon_Code':
            coupon,

        'Is_First_Order':
            random.choices(

                [
                    'Yes',
                    'No',
                    'yes',
                    'no',
                    1,
                    0,
                    None
                ],

                weights=[
                    15,
                    60,
                    5,
                    8,
                    5,
                    5,
                    2
                ]

            )[0]
    })


# ═══════════════════════════════════════════════════════════════════════════════
# 13. CREATE DATAFRAME
# ═══════════════════════════════════════════════════════════════════════════════

df = pd.DataFrame(
    orders
)


# ═══════════════════════════════════════════════════════════════════════════════
# 14. ADD INTENTIONAL DUPLICATES AND BLANK ROWS
# ═══════════════════════════════════════════════════════════════════════════════

dupes = df.sample(
    150,
    random_state=44
)

blanks = pd.DataFrame(

    [
        {
            c: None
            for c in df.columns
        }

        for _ in range(20)
    ]

)

df = pd.concat(
    [
        df,
        dupes,
        blanks
    ],

    ignore_index=True
)


# ═══════════════════════════════════════════════════════════════════════════════
# 15. SAVE RAW ORDERS DATA
# ═══════════════════════════════════════════════════════════════════════════════

df.to_csv(
    'KartZone_Orders_V2.csv',
    index=False
)


# ═══════════════════════════════════════════════════════════════════════════════
# 16. VALIDATION OF GENERATED DATA
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("  GENERATED ORDERS VALIDATION")
print("=" * 70)

print(
    f"\nRows generated       : {len(df):,}"
)

print(
    f"Columns              : {len(df.columns)}"
)

print(
    f"Null values          : {df.isnull().sum().sum():,}"
)

print(
    f"Exact duplicate rows : {df.duplicated().sum():,}"
)


# Check Customer IDs before intentional orphan rows
customer_valid_check = (
    df['Customer_ID'].isin(CUSTOMER_IDS)
)

print(
    f"Valid Customer IDs   : "
    f"{customer_valid_check.sum():,}"
)

print(
    f"Invalid Customer IDs : "
    f"{(~customer_valid_check).sum():,}"
)


# Check Product IDs before intentional orphan rows
product_valid_check = (
    df['Product_ID'].isin(PRODUCT_IDS)
)

print(
    f"Valid Product IDs    : "
    f"{product_valid_check.sum():,}"
)

print(
    f"Invalid Product IDs  : "
    f"{(~product_valid_check).sum():,}"
)


# ═══════════════════════════════════════════════════════════════════════════════
# 17. PRINT DATA QUALITY ISSUES
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("  DATA QUALITY ISSUES EMBEDDED")
print("=" * 70)

print("  ✓ Mixed date formats")
print("  ✓ Currency symbols in Final_Amount")
print("  ✓ Inconsistent Order_Status")
print("  ✓ Inconsistent Payment_Mode")
print("  ✓ Inconsistent Category names")
print("  ✓ Negative Final_Amount")
print("  ✓ Invalid discount values")
print("  ✓ Profit as null / negative / string")
print("  ✓ Quantity as int / float / string / null")
print("  ✓ Approximately 2% orphan Customer_IDs")
print("  ✓ Approximately 2% orphan Product_IDs")
print("  ✓ 150 duplicate order rows")
print("  ✓ 20 fully blank rows")


# ═══════════════════════════════════════════════════════════════════════════════
# 18. HIDDEN BUSINESS PATTERNS
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("  HIDDEN BUSINESS PATTERNS")
print("=" * 70)

print("  ✓ Oct-Dec festive spike")
print("  ✓ Fashion returns increase when delivery > 5 days")
print("  ✓ COD orders have higher cancellation rate")
print("  ✓ Electronics discounts > 25% increase return tendency")
print("  ✓ UPI is the dominant payment method")
print("  ✓ Product IDs are linked to the actual Product master")
print("  ✓ Customer IDs are linked to the actual Customer master")


# ═══════════════════════════════════════════════════════════════════════════════
# 19. FEATURE ENGINEERING SUPPORTED
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("  FEATURE ENGINEERING SUPPORTED")
print("=" * 70)

print("  ✓ Profit")
print("  ✓ Profit_Margin_Pct")
print("  ✓ Is_Return")
print("  ✓ Is_Cancelled")
print("  ✓ Is_Delivered")
print("  ✓ High_Discount_Flag")
print("  ✓ Order_Value_Band")
print("  ✓ Day_of_Week")
print("  ✓ Month")
print("  ✓ Quarter")
print("  ✓ Is_Weekend")
print("  ✓ Is_Festive_Season")
print("  ✓ Coupon_Used")
print("  ✓ Revenue_Contribution_Pct")


# ═══════════════════════════════════════════════════════════════════════════════
# 20. SAMPLE OUTPUT
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("  SAMPLE ORDERS")
print("=" * 70)

print(
    df.head()
)


print("\n" + "=" * 70)
print("  ORDERS GENERATION COMPLETE")
print("=" * 70)

print(
    "\nSaved → KartZone_Orders_V2.csv ✓"
)



  KARTZONE INDIA — ORDERS TABLE GENERATOR

Loading cleaned master tables...
Valid Customer IDs loaded : 1,000
Valid Product IDs loaded  : 800
Product category lookup created.

Generating 10,000 order records...

  GENERATED ORDERS VALIDATION

Rows generated       : 10,170
Columns              : 21
Null values          : 16,628
Exact duplicate rows : 169
Valid Customer IDs   : 9,929
Invalid Customer IDs : 241
Valid Product IDs    : 9,956
Invalid Product IDs  : 214

  DATA QUALITY ISSUES EMBEDDED
  ✓ Mixed date formats
  ✓ Currency symbols in Final_Amount
  ✓ Inconsistent Order_Status
  ✓ Inconsistent Payment_Mode
  ✓ Inconsistent Category names
  ✓ Negative Final_Amount
  ✓ Invalid discount values
  ✓ Profit as null / negative / string
  ✓ Quantity as int / float / string / null
  ✓ Approximately 2% orphan Customer_IDs
  ✓ Approximately 2% orphan Product_IDs
  ✓ 150 duplicate order rows
  ✓ 20 fully blank rows

  HIDDEN BUSINESS PATTERNS
  ✓ Oct-Dec festive spike
  ✓ Fashion returns inc